In [1]:
import sys
from pathlib import Path

# Garante que o pacote do projeto (src/spark_delta) seja encontrado mesmo que o
# kernel não tenha sido instalado com `uv sync` (fallback para src/).
try:
    import spark_delta  # noqa: F401
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
    sys.path.insert(0, str(Path.cwd().resolve() / "src"))

from spark_delta.config import settings
from spark_delta.spark_session import get_spark, WAREHOUSE_DIR
from spark_delta.storage import s3_client, ensure_bucket

s3 = s3_client()
for bucket in (settings.bronze_bucket, settings.warehouse_bucket):
    ensure_bucket(s3, bucket)

spark = get_spark("csv_to_delta")
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| warehouse:", WAREHOUSE_DIR)
spark

your 131072x1 screen size is bogus. expect trouble
26/10/13 20:36:12 WARN Utils: Your hostname, Maicou resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/13 20:36:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/maicou/projeto2/spark-delta/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/maicou/.ivy2/cache
The jars for the packages stored in: /home/maicou/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2fec5b0f-e35a-4c03-a224-83bbec6f751a;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (256ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java-s

Spark 3.5.3 | warehouse: s3a://warehouse/spark-warehouse


In [2]:
from spark_delta.storage import list_objects

csv_files = sorted(o["Key"] for o in list_objects(s3, settings.landing_bucket) if o["Key"].endswith(".csv"))
print(f"{len(csv_files)} CSVs em s3://{settings.landing_bucket}/")
for f in csv_files:
    print("  -", f)

7 CSVs em s3://landing-zone/
  - categoria.csv
  - cliente.csv
  - estoque.csv
  - fornecedor.csv
  - item_venda.csv
  - produto.csv
  - venda.csv


In [3]:
from pyspark.sql import functions as F
from spark_delta.schemas import SCHEMAS

resumo = []
for csv_file in csv_files:
    tabela = csv_file.removesuffix(".csv")
    origem = f"s3a://{settings.landing_bucket}/{csv_file}"
    destino = f"s3a://{settings.bronze_bucket}/{tabela}"

    leitor = (spark.read
              .option("header", "true")
              .option("dateFormat", "yyyy-MM-dd")
              .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
              .option("mode", "FAILFAST"))          # erro de conversão interrompe (não vira NULL)
    if tabela in SCHEMAS:
        df = leitor.schema(SCHEMAS[tabela]).csv(origem)
    else:
        df = leitor.option("inferSchema", "true").csv(origem)

    df = df.withColumn("_dt_ingestao", F.current_timestamp())

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .save(destino))

    qtd = spark.read.format("delta").load(destino).count()
    resumo.append((tabela, qtd, destino))
    print(f"  {tabela:<12} {qtd:>6} registros -> {destino}")

print(f"\n{len(resumo)} tabelas Delta gravadas no bucket [{settings.bronze_bucket}].")

  categoria         8 registros -> s3a://bronze/categoria


  cliente        1000 registros -> s3a://bronze/cliente
  estoque         120 registros -> s3a://bronze/estoque
  fornecedor       25 registros -> s3a://bronze/fornecedor
  item_venda    10964 registros -> s3a://bronze/item_venda
  produto         120 registros -> s3a://bronze/produto
  venda          5000 registros -> s3a://bronze/venda

7 tabelas Delta gravadas no bucket [bronze].


In [4]:
from delta.tables import DeltaTable

for tabela, qtd, caminho in resumo:
    print(f"  {tabela:<12} Delta={DeltaTable.isDeltaTable(spark, caminho)} | {qtd} registros")

  categoria    Delta=True | 8 registros
  cliente      Delta=True | 1000 registros
  estoque      Delta=True | 120 registros
  fornecedor   Delta=True | 25 registros
  item_venda   Delta=True | 10964 registros
  produto      Delta=True | 120 registros
  venda        Delta=True | 5000 registros


In [5]:
# Schema gravado na camada bronze (tipos corretos e CPF como texto)
spark.read.format("delta").load(f"s3a://{settings.bronze_bucket}/cliente").printSchema()
spark.read.format("delta").load(f"s3a://{settings.bronze_bucket}/cliente").show(5, truncate=False)

root
 |-- cd_cliente: integer (nullable = true)
 |-- nm_cliente: string (nullable = true)
 |-- cpf: string (nullable = true)
 |-- email: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- dt_cadastro: date (nullable = true)
 |-- _dt_ingestao: timestamp (nullable = true)

+----------+--------------------------+-----------+-----------------------------+---------+---+-----------+--------------------------+
|cd_cliente|nm_cliente                |cpf        |email                        |cidade   |uf |dt_cadastro|_dt_ingestao              |
+----------+--------------------------+-----------+-----------------------------+---------+---+-----------+--------------------------+
|1         |Natália Fernandes Ribeiro |17839084700|natalia.ribeiro1@email.com   |Curitiba |PR |2023-09-08 |2026-10-13 20:37:54.117423|
|2         |Rodrigo Carvalho Gomes    |17711592124|rodrigo.gomes2@email.com     |São Paulo|SP |2024-07-02 |2026-10-13 20:37:54.117423|
|3

In [6]:
# Estrutura física de uma tabela Delta: arquivos Parquet + pasta _delta_log
for o in list_objects(s3, settings.bronze_bucket, prefix="produto/"):
    print(f"  {o['Key']:<90} {o['Size']:>8} bytes")

  produto/_delta_log/00000000000000000000.json                                                   1986 bytes
  produto/_delta_log/_commits/                                                                      0 bytes
  produto/part-00000-061067b9-58c4-446a-8b55-d1c31ef610cc-c000.snappy.parquet                    5324 bytes


In [7]:
spark.stop()
print("SparkSession finalizada.")

SparkSession finalizada.
